# Lab 13B: Self-supervised learning with SimCLR

This is the second notebook of Lab 13 (see 13A for an overview).

**The labelling bottleneck** (Lecture 13, Section 2.1). Images are cheap; labels are expensive. Imagine we
have 15,000 photos of cats, dogs and horses, but time to label only 150 of them. A network trained on 150
labelled images does not learn much. **Self-supervised learning** first learns features from *all* the images,
without any labels, and then uses the few labels only for a simple classifier on top of those features.

We use **SimCLR** (Section 3.3): a network learns to recognise two differently augmented versions of the
same image as "the same", and different images as "different". Nobody tells it what a cat is.

| Part | You will… |
|---|---|
| 1. Two views of one image | see the augmentations that create the training pairs |
| 2. Why "pull the two views together" is not enough | watch a network find a useless shortcut (*collapse*) |
| 3. The contrastive loss | turn "find your twin" into a classification problem, and write its two key lines |
| 4. Pre-training without labels | train the encoder on 15,000 unlabelled images |
| 5. Few labels | compare self-supervised features with training from scratch, using 150 and 1,500 labels |
| 6. Exam-style questions | check your understanding |

## Setup

In [ ]:
import math
import random
import time

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

import os
import zipfile
from pathlib import Path

import gdown
from PIL import Image

# The cat, dog and horse images of CIFAR-10 (32 × 32 pixels), downloaded from Google Drive:
# train/{cat,dog,horse}: 5,000 images each; test/{cat,dog,horse}: 1,000 images each
CLASSES = ['cat', 'dog', 'horse']

if not Path('train').exists():
    if not Path('cifar_cat_dog_horse.zip').exists():
        file_id = '1wvqLLFioimWDLfl7oqzCZi-q-tKDfXVg'
        gdown.download(f'https://drive.google.com/uc?id={file_id}', 'cifar_cat_dog_horse.zip', quiet=False)
    with zipfile.ZipFile('cifar_cat_dog_horse.zip') as z:
        z.extractall('.')


def load_split(split):
    xs, ys = [], []
    for label, name in enumerate(CLASSES):
        files = sorted(Path(split, name).glob('*.jpg'))
        xs += [np.array(Image.open(f).convert('RGB')) for f in files]
        ys += [label] * len(files)
    x = torch.tensor(np.stack(xs)).permute(0, 3, 1, 2).float() / 255   # (N, 3, 32, 32) in [0, 1]
    y = torch.tensor(ys)
    perm = torch.randperm(len(y), generator=torch.Generator().manual_seed(0))   # mix the classes
    return x[perm], y[perm]


X_train, Y_train = load_split('train')
X_test, Y_test = load_split('test')
MEAN = X_train.mean((0, 2, 3), keepdim=True)
STD = X_train.std((0, 2, 3), keepdim=True)
print(f'{len(X_train)} training images, {len(X_test)} test images')




def simclr_augment(x, colour=True):
    # Two calls give two different random "views" of the same images (all on the GPU, a different random choice per image):
    # random crop (20-100% of the area) and resize, horizontal flip, colour jitter (80% of the images), grey-scale (20%)
    B, dev = x.shape[0], x.device
    area = torch.empty(B, device=dev).uniform_(0.2, 1.0)
    ratio = torch.exp(torch.empty(B, device=dev).uniform_(math.log(3 / 4), math.log(4 / 3)))
    w, h = (area * ratio).sqrt().clamp(max=1), (area / ratio).sqrt().clamp(max=1)
    cx, cy = (torch.rand(B, device=dev) * 2 - 1) * (1 - w), (torch.rand(B, device=dev) * 2 - 1) * (1 - h)
    flip = torch.where(torch.rand(B, device=dev) < 0.5, -1.0, 1.0)
    theta = torch.zeros(B, 2, 3, device=dev)
    theta[:, 0, 0], theta[:, 0, 2], theta[:, 1, 1], theta[:, 1, 2] = w * flip, cx, h, cy
    x = F.grid_sample(x, F.affine_grid(theta, x.shape, align_corners=False), align_corners=False, padding_mode='reflection')
    if colour:
        r = lambda: 1 + (torch.rand(B, 1, 1, 1, device=dev) * 2 - 1) * 0.4
        y = x * r()                                                              # brightness
        m = y.mean((1, 2, 3), keepdim=True); y = (y - m) * r() + m                # contrast
        g = y.mean(1, keepdim=True); y = (y - g) * r() + g                       # saturation
        apply = (torch.rand(B, 1, 1, 1, device=dev) < 0.8).float()
        x = apply * y + (1 - apply) * x
        grey = (torch.rand(B, 1, 1, 1, device=dev) < 0.2).float()
        x = grey * x.mean(1, keepdim=True).expand_as(x) + (1 - grey) * x
    return x.clamp(0, 1)


def normalise(x):
    return (x - MEAN.to(x.device)) / STD.to(x.device)


def conv_block(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
                         nn.Conv2d(c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(), nn.MaxPool2d(2))


def Encoder(c=32):
    # A small CNN: image (3, 32, 32) -> representation h with 4c = 128 numbers
    return nn.Sequential(conv_block(3, c), conv_block(c, 2 * c), conv_block(2 * c, 4 * c), nn.AdaptiveAvgPool2d(1), nn.Flatten())


def ProjectionHead(d=128, d_out=64):
    # SimCLR's projection head: h -> z, used only for the contrastive loss
    return nn.Sequential(nn.Linear(d, d), nn.ReLU(), nn.Linear(d, d_out))


X_train_dev, X_test_dev = X_train.to(device), X_test.to(device)

---
# Part 1 — Two views of one image

SimCLR creates its training pairs by **augmentation**: every image is augmented twice, with different random
crops, flips and colour changes. The two results are two **views** of the same image.

In [ ]:
idx = torch.arange(8)
x = X_train_dev[idx]
set_seed(0)
rows = [x, simclr_augment(x), simclr_augment(x)]
fig, axs = plt.subplots(3, 8, figsize=(12, 4.8))
for r, (imgs, name) in enumerate(zip(rows, ['original', 'view 1', 'view 2'])):
    for c in range(8):
        axs[r, c].imshow(imgs[c].permute(1, 2, 0).cpu()); axs[r, c].axis('off')
    axs[r, 0].set_title(name, loc='left', fontsize=10)
plt.tight_layout(); plt.show()

### Questions 1

1. What do the two views of an image have in common, and what is different?
2. A network that gives the two views the same representation must ignore exactly the things the augmentation changes.
   Which properties of an image does it learn to ignore? Why is that useful for recognising cats, dogs and horses?
3. The SimCLR paper found that **cropping together with colour changes** works much better than either alone. Why
   might cropping alone not be enough? (Hint: two crops of the same photo usually have…)

---
# Part 2 — Why "pull the two views together" is not enough

The obvious loss: make the embeddings $\mathbf{z}_i$ and $\mathbf{z}_j$ of the two views of an image as similar
as possible, for example by minimising $1 - \text{sim}(\mathbf{z}_i, \mathbf{z}_j)$, where sim is the cosine
similarity (the normalised dot product from 12A).

✍️ **Prediction:** we train an encoder with this loss for 300 steps. What will happen to the loss? Will the
embeddings of *different* images become similar or different?

Below, we track two numbers: the loss, and the average similarity between the embeddings of **different**
images in the batch.

In [ ]:
set_seed(0)
enc, head = Encoder().to(device), ProjectionHead().to(device)
opt = torch.optim.Adam(list(enc.parameters()) + list(head.parameters()), lr=1e-3)
naive_loss, others = [], []
for step in range(300):
    x = X_train_dev[torch.randint(0, len(X_train), (256,), device=device)]
    z1 = F.normalize(head(enc(normalise(simclr_augment(x)))), dim=1)
    z2 = F.normalize(head(enc(normalise(simclr_augment(x)))), dim=1)
    loss = (1 - (z1 * z2).sum(1)).mean()                     # 1 - sim(z_i, z_j), averaged over the batch
    opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        s = z1 @ z1.T                                        # similarities between different images (off the diagonal)
        others.append(((s.sum() - s.diag().sum()) / (256 * 255)).item())
    naive_loss.append(loss.item())

fig, axs = plt.subplots(1, 2, figsize=(10, 3))
axs[0].plot(naive_loss); axs[0].set_title('loss: 1 - sim(two views)'); axs[0].set_xlabel('step')
axs[1].plot(others); axs[1].set_title('average sim between DIFFERENT images'); axs[1].set_xlabel('step'); axs[1].set_ylim(-0.1, 1.05)
plt.tight_layout(); plt.show()

### Questions 2

1. What happened to the loss and to the similarity between different images?
2. Explain the shortcut the network found. Why does it give (almost) zero loss? Is the representation useful?
3. What does a good loss need besides "pull the two views together"?

---
# Part 3 — The contrastive loss: "find your twin"

To prevent collapse, the loss must also **push different images apart**. SimCLR does this with a neat trick:
it turns the problem into a **classification** problem, so we can use the cross-entropy loss we have used all
course (and whose meaning the lecture explains in Section 1).

**The task.** A batch has $N$ images, so $2N$ views. Take one view $i$. Among the other $2N - 1$ views, exactly
one is its **twin** $j$ (the other view of the same image); the rest are views of other images. The network must
pick out the twin: a $(2N-1)$-way classification problem, whose "label" we know without any human labelling.

**Step 1: scores.** The score of candidate $k$ is the cosine similarity of the embeddings, divided by a
**temperature** $\tau$:

$$s_{ik} = \frac{\text{sim}(\mathbf{z}_i, \mathbf{z}_k)}{\tau}$$

A cosine similarity lies between −1 and 1, which is too narrow a range for a confident softmax. Dividing by a small $\tau$
(we use 0.2) stretches it: this is the same **sharpness** knob as $s$ in the database lookup of 12A, just written as a division.

**Step 2: probabilities.** A softmax over all candidates gives the probability that $k$ is the twin of $i$:

$$p(k \mid i) = \frac{\exp(s_{ik})}{\sum_{k' \neq i} \exp(s_{ik'})}$$

**Step 3: cross-entropy.** The label is the twin $j$, so the loss is the cross-entropy with a one-hot target,
$-\log p(j \mid i)$. Written out, this is the **NT-Xent** loss of the lecture (Normalised Temperature-scaled
cross-entropy):

$$\mathcal{L}_i = -\log \frac{\exp\!\left(\text{sim}(\mathbf{z}_i, \mathbf{z}_j)/\tau\right)}{\sum_{k \neq i} \exp\!\left(\text{sim}(\mathbf{z}_i, \mathbf{z}_k)/\tau\right)}$$

averaged over all $2N$ views.

**Why it cannot collapse.** If all embeddings were the same, all scores would be equal, the softmax would give every
candidate probability $1/(2N-1)$, and the loss would be $\log(2N-1)$: its *worst* value for a network that cannot tell
anything apart. To lower the loss, the twin must be **more similar than all the other images**, so the embeddings
of different images must differ. The other images in the batch act as **negatives**.

### 🛠️ TASK 1: The NT-Xent loss

The code stacks both views into one matrix `z` of $2N$ rows: row $i < N$ is view 1 of image $i$, row $i + N$ is
view 2 of the same image. So the twin of row $i$ is row $i + N$, and vice versa.

**🔗 Theory ↔ code**

| Lecture | Code | Shape |
|---|---|---|
| $\mathbf{z}$ scaled to length 1, so that dot products are cosine similarities | `z = F.normalize(..., dim=1)` (given) | (2N, d) |
| all scores $s_{ik} = \text{sim}(\mathbf{z}_i, \mathbf{z}_k)/\tau$ at once | `z @ z.T / temperature` | (2N, 2N) |
| $k \neq i$: a view is not a candidate for its own twin | the diagonal is set to $-\infty$, so $\exp = 0$ (given) | |
| the label of row $i$: the index of its twin | `labels` (given) | (2N,) |
| softmax + $-\log p(\text{twin})$, averaged | `F.cross_entropy(scores, labels)` | |

In [ ]:
def nt_xent_loss(z1, z2, temperature=0.2):
    # z1, z2: (N, d) embeddings of view 1 and view 2 of the same N images
    N = z1.shape[0]
    z = F.normalize(torch.cat([z1, z2]), dim=1)                     # (2N, d), every row has length 1
    scores = ???                                                    # all cosine similarities / temperature   (2N, 2N)
    scores = scores.masked_fill(torch.eye(2 * N, dtype=torch.bool, device=z.device), float('-inf'))   # k != i
    labels = torch.cat([torch.arange(N, 2 * N), torch.arange(N)]).to(z.device)                        # the twin of each row
    loss = ???                                                      # cross-entropy: find the twin
    accuracy = (scores.argmax(1) == labels).float().mean()          # how often the twin gets the highest score
    return loss, accuracy

In [ ]:
# Test for Task 1
# (1) Compare with a direct implementation of the formula, row by row
_z1, _z2 = torch.randn(5, 8), torch.randn(5, 8)
_loss, _ = nt_xent_loss(_z1, _z2, temperature=0.5)
_z = F.normalize(torch.cat([_z1, _z2]), dim=1)
_ref = 0
for _i in range(10):
    _j = (_i + 5) % 10
    _s = torch.stack([_z[_i] @ _z[_k] / 0.5 for _k in range(10) if _k != _i])
    _ref += -(_z[_i] @ _z[_j] / 0.5 - torch.logsumexp(_s, 0))
assert torch.allclose(_loss, _ref / 10, atol=1e-5), f'the loss {_loss.item():.4f} differs from the formula {(_ref / 10).item():.4f}'
# (2) Collapse: identical embeddings give the worst loss, log(2N - 1)
_loss_c, _ = nt_xent_loss(torch.ones(5, 8), torch.ones(5, 8))
assert abs(_loss_c.item() - math.log(9)) < 1e-4, 'if all embeddings are equal, the loss should be log(2N - 1)'
print('✅ Task 1 passed')
print(f'Loss for collapsed embeddings with N = 256: log(511) = {math.log(511):.2f}')

---
# Part 4 — Pre-training without labels

We now train a fresh encoder and projection head with the NT-Xent loss, on all 15,000 training images.
**No labels are used.** Besides the loss, we track the *contrastive accuracy*: how often a view's twin
gets the highest score among the 511 candidates (chance: 1/511 ≈ 0.2%). This takes a few minutes on a GPU.

In [ ]:
def pretrain_simclr(steps=3000, batch_size=256, colour=True, seed=0):
    set_seed(seed)
    enc, head = Encoder().to(device), ProjectionHead().to(device)
    opt = torch.optim.Adam(list(enc.parameters()) + list(head.parameters()), lr=1e-3)
    history = []
    t0 = time.time()
    for step in range(steps):
        x = X_train_dev[torch.randint(0, len(X_train), (batch_size,), device=device)]   # NO labels
        z1 = head(enc(normalise(simclr_augment(x, colour))))
        z2 = head(enc(normalise(simclr_augment(x, colour))))
        loss, acc = nt_xent_loss(z1, z2)
        opt.zero_grad(); loss.backward(); opt.step()
        history.append((loss.item(), acc.item()))
        if step % 500 == 0 or step == steps - 1:
            print(f'step {step}: loss {loss.item():.2f}, contrastive accuracy {acc.item():.0%}  ({time.time() - t0:.0f} s)')
    return enc, np.array(history)


encoder, history = pretrain_simclr()
fig, axs = plt.subplots(1, 2, figsize=(10, 3))
axs[0].plot(history[:, 0]); axs[0].axhline(math.log(511), color='grey', ls='--', label='collapse: log(511)')
axs[0].set_title('NT-Xent loss'); axs[0].set_xlabel('step'); axs[0].legend()
axs[1].plot(history[:, 1]); axs[1].set_title('contrastive accuracy'); axs[1].set_xlabel('step')
plt.tight_layout(); plt.show()

### Questions 4

1. Where does the loss start, and why? (Compare with the dashed line.)
2. The contrastive accuracy ends far above chance. What has the encoder learned to do? Has it learned anything about
   cats, dogs and horses? We have not told it they exist.

---
# Part 5 — Few labels: does the pre-training help?

Now we pretend we have labels for only **30** images (10 per class), **150** (1%) or **1,500** (10%), and compare
three ways of using them:

| Method | What is trained on the labelled images |
|---|---|
| **SimCLR + linear probe** | only a linear classifier on the frozen encoder's representation $\mathbf{h}$ (the standard SSL evaluation, Section 2.2) |
| **Random encoder + linear probe** | the same, on an untrained encoder (a control: how good are random features?) |
| **From scratch** | the whole CNN (same architecture), with normal augmentation, as in earlier labs |

We use the representation $\mathbf{h}$ *before* the projection head; the head is thrown away after pre-training
(Section 3.3, finding (2)).

✍️ **Prediction:** which method wins with 30 labels? With 1,500?

In [ ]:
@torch.no_grad()
def features(enc, X):
    enc.eval()
    return torch.cat([enc(normalise(X[k:k + 500])) for k in range(0, len(X), 500)])


def linear_probe(F_train, y_train, F_test, steps=500):
    # A linear classifier (softmax regression) on frozen features
    mu, sd = F_train.mean(0), F_train.std(0) + 1e-6
    clf = nn.Linear(F_train.shape[1], 3).to(device)
    opt = torch.optim.Adam(clf.parameters(), lr=1e-2, weight_decay=1e-4)
    for _ in range(steps):
        loss = F.cross_entropy(clf((F_train - mu) / sd), y_train)
        opt.zero_grad(); loss.backward(); opt.step()
    return (clf((F_test - mu) / sd).argmax(1) == Y_test.to(device)).float().mean().item()


def train_from_scratch(idx, steps=800):
    set_seed(1)
    model = nn.Sequential(Encoder(), nn.Linear(128, 3)).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    X, Y = X_train_dev[idx], Y_train.to(device)[idx]
    for _ in range(steps):
        b = torch.randint(0, len(idx), (min(128, len(idx)),), device=device)
        x = X[b]
        x = torch.where(torch.rand(len(x), 1, 1, 1, device=device) < 0.5, x.flip(3), x)      # flips
        i, j = torch.randint(0, 9, (2,))
        x = F.pad(x, (4, 4, 4, 4), mode='reflect')[:, :, i:i + 32, j:j + 32]                  # shifts
        loss = F.cross_entropy(model(normalise(x)), Y[b])
        opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        return (features(model, X_test_dev).argmax(1) == Y_test.to(device)).float().mean().item()


set_seed(0)
F_train, F_test = features(encoder, X_train_dev), features(encoder, X_test_dev)
random_encoder = Encoder().to(device)
R_train, R_test = features(random_encoder, X_train_dev), features(random_encoder, X_test_dev)
perm = torch.randperm(len(X_train), generator=torch.Generator().manual_seed(0)).to(device)
Y_dev = Y_train.to(device)

LABELS = [30, 150, 1500]
results = {'SimCLR + linear probe': [], 'random encoder + linear probe': [], 'from scratch': []}
for n in LABELS:
    idx = perm[:n]
    results['SimCLR + linear probe'].append(linear_probe(F_train[idx], Y_dev[idx], F_test))
    results['random encoder + linear probe'].append(linear_probe(R_train[idx], Y_dev[idx], R_test))
    results['from scratch'].append(train_from_scratch(idx))
    print(f'{n} labels: ' + ',  '.join(f'{k} {v[-1]:.0%}' for k, v in results.items()))

x = np.arange(len(LABELS))
plt.figure(figsize=(7, 4))
for k, (name, acc) in enumerate(results.items()):
    bars = plt.bar(x + (k - 1) * 0.27, acc, 0.27, label=name)
    plt.bar_label(bars, [f'{a:.0%}' for a in acc], fontsize=8)
plt.axhline(1 / 3, color='grey', ls='--', label='chance')
plt.xticks(x, [f'{n:,} labelled images' for n in LABELS]); plt.ylabel('test accuracy'); plt.ylim(0, 1)
plt.legend(loc='upper left', fontsize=8); plt.title('Few labels'); plt.show()

### Questions 5

1. Which method is best with 30 labels? With 1,500? How does the benefit of pre-training change as the number of labels grows?
2. The linear probe trains only $128 \times 3 + 3 = 387$ parameters on the labelled images. Why can it beat a whole CNN trained
   from scratch on the same images?
3. Why is the random encoder a useful control?
4. Lecture Exercise 3(e): a hospital has five million unlabelled chest X-rays and a few hundred labelled ones. Describe how
   you would use SimCLR. What would you use as augmentations? (Careful: is a horizontally flipped chest X-ray still a
   realistic X-ray?)

### 🔧 Optional knob: pre-training without colour changes

If you have time: pre-train again **without** the colour changes (crops and flips only), and repeat the linear
probe with 150 labels. Compare with your answer to Question 1.3.

In [ ]:
encoder_nc, history_nc = pretrain_simclr(colour=False)
Fn_train, Fn_test = features(encoder_nc, X_train_dev), features(encoder_nc, X_test_dev)
idx = perm[:150]
print(f'150 labels, linear probe: with colour changes {results["SimCLR + linear probe"][LABELS.index(150)]:.0%}, '
      f'without {linear_probe(Fn_train[idx], Y_dev[idx], Fn_test):.0%}')
print(f'Final contrastive accuracy: with colour changes {history[-50:, 1].mean():.0%}, without {history_nc[-50:, 1].mean():.0%}')

---
# Part 6 — Exam-style questions

**Q1.** In self-supervised learning, where do the labels for the pretext task come from?

* (a) From human annotators.
* (b) From the data itself, for example from the augmentations or transformations that were applied.
* (c) From a pre-trained classifier.
* (d) There are no labels and no loss.

**Q2.** What is "collapse" in contrastive learning?

* (a) The loss diverges to infinity.
* (b) The encoder maps every input to (almost) the same embedding, which makes all views trivially similar.
* (c) The batch size becomes too small.
* (d) The learning rate drops to zero.

**Q3.** In SimCLR's NT-Xent loss for a batch of $N$ images, what are the negatives for one view?

* (a) Its augmented twin.
* (b) Images of other classes, selected using the labels.
* (c) The other $2N - 2$ views in the batch, which come from different images.
* (d) Random noise images.

**Q4.** How is a self-supervised representation typically evaluated?

* (a) By its accuracy on the pretext task.
* (b) By the value of the contrastive loss.
* (c) By freezing the encoder and training only a linear classifier on top of it for a downstream task with labels.
* (d) By the number of parameters of the encoder.

**Q5.** CLIP is trained with the same kind of loss as SimCLR. What are its positive pairs?

* (a) Two augmented views of the same image.
* (b) An image and the text caption that belongs to it.
* (c) Two images of the same class.
* (d) Two captions of the same image.